In [1]:
# ==========================================================
# GSHARE BRANCH PREDICTOR WITH FEDERATED LEARNING
# ==========================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple, Optional

# ==========================================================
# SETTINGS
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "6_clients"
FL_ROUNDS = 50
BATCH_SIZE = 256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================================
# GSHARE CONFIGURATION
# ==========================================================
class GShareConfig:
    def __init__(self, table_size: int = 4096, history_bits: int = 12):
        """
        GShare configuration
        
        Args:
            table_size: Number of entries in prediction table (must be power of 2)
            history_bits: Number of bits in global history register
        """
        self.table_size = table_size
        self.history_bits = history_bits
        self.counter_bits = 2  # 2-bit saturating counters (0-3)
        
        # Table size must be power of 2 for efficient indexing
        assert (table_size & (table_size - 1)) == 0, "Table size must be power of 2"
        
    def get_total_storage_bits(self) -> int:
        """Calculate total storage in bits"""
        # Prediction table: table_size * counter_bits
        # Global history register: history_bits
        return (self.table_size * self.counter_bits) + self.history_bits
    
    def get_total_storage_bytes(self) -> int:
        """Calculate total storage in bytes"""
        return self.get_total_storage_bits() // 8
    
    def get_total_storage_kb(self) -> float:
        """Calculate total storage in KB"""
        return self.get_total_storage_bits() / (8 * 1024)

# ==========================================================
# DATA LOADING
# ==========================================================
def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values.astype(np.int64)  # Branch PC and history
    y = df.iloc[:, -1].values.astype(np.int64)   # Branch outcome (0/1)
    return X, y

def make_loader(X, y, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

# ==========================================================
# GSHARE PREDICTOR
# ==========================================================
class GSharePredictor:
    """
    GShare Branch Predictor
    Combines global history with branch PC using XOR
    Uses 2-bit saturating counters for prediction
    """
    def __init__(self, config: GShareConfig):
        self.config = config
        
        # Prediction table: 2-bit saturating counters
        # Values: 0 = strongly not taken, 1 = weakly not taken,
        #         2 = weakly taken, 3 = strongly taken
        self.prediction_table = [2] * config.table_size  # Start with weakly taken
        
        # Global history register (shift register)
        self.global_history = 0
        self.history_mask = (1 << config.history_bits) - 1
        
        # Statistics
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0
        
    def _get_index(self, pc: int) -> int:
        """
        Compute index into prediction table using GShare indexing:
        index = (PC XOR global_history) % table_size
        """
        # XOR PC with global history
        if self.config.history_bits > 0:
            # Use only the relevant bits of PC
            pc_bits = pc & self.history_mask
            index = pc_bits ^ self.global_history
        else:
            index = pc
        
        return index % self.config.table_size
    
    def predict(self, pc: int) -> bool:
        """
        Predict branch direction
        Returns: True = taken, False = not taken
        """
        self.predictions += 1
        
        idx = self._get_index(pc)
        counter = self.prediction_table[idx]
        
        # Prediction: taken if counter >= 2 (weakly or strongly taken)
        return counter >= 2
    
    def update(self, pc: int, taken: bool):
        """
        Update predictor with actual branch outcome
        """
        idx = self._get_index(pc)
        counter = self.prediction_table[idx]
        
        # Update saturating counter
        if taken:
            if counter < 3:
                counter += 1
        else:
            if counter > 0:
                counter -= 1
        
        self.prediction_table[idx] = counter
        
        # Update global history (shift left, add new outcome)
        self.global_history = ((self.global_history << 1) | (1 if taken else 0))
        self.global_history &= self.history_mask
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """
        Make prediction and update with actual outcome
        Returns: prediction made before update
        """
        pred = self.predict(pc)
        
        # Update statistics
        if pred == actual:
            self.correct_predictions += 1
        else:
            self.mispredictions += 1
        
        # Update predictor state
        self.update(pc, actual)
        
        return pred
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy percentage"""
        if self.predictions == 0:
            return 0.0
        return (self.correct_predictions / self.predictions) * 100
    
    def get_mpki(self, instructions: int) -> float:
        """Calculate MPKI (Mispredictions Per Kilo Instructions)"""
        if instructions == 0:
            return 0.0
        return (self.mispredictions / instructions) * 1000
    
    def reset(self):
        """Reset predictor state"""
        self.prediction_table = [2] * self.config.table_size
        self.global_history = 0
        self.predictions = 0
        self.mispredictions = 0
        self.correct_predictions = 0

# ==========================================================
# GSHARE MODEL WRAPPER FOR FEDERATED LEARNING
# ==========================================================
class GShareModel(nn.Module):
    """
    Wrapper to make GShare compatible with PyTorch training loop
    Note: This is not a neural network; we're using PyTorch's structure
    for federated learning orchestration only
    """
    def __init__(self, config: GShareConfig):
        super().__init__()
        self.config = config
        self.predictor = GSharePredictor(config)
        
        # For compatibility with PyTorch optimizer
        self.parameters_list = nn.ParameterList([
            nn.Parameter(torch.tensor(0.0))  # Dummy parameter
        ])
        
    def forward(self, x):
        """
        Forward pass for batch prediction
        x: batch of branch PCs
        """
        batch_size = x.size(0)
        predictions = []
        
        for i in range(batch_size):
            pc = x[i, 0].item()  # Assuming PC is first column
            pred = self.predictor.predict(pc)
            predictions.append(1.0 if pred else 0.0)
        
        return torch.tensor(predictions, device=x.device)
    
    def update(self, pc: int, taken: bool):
        """Update predictor state"""
        self.predictor.update(pc, taken)
    
    def predict_and_update(self, pc: int, actual: bool) -> bool:
        """Predict and update with actual outcome"""
        return self.predictor.predict_and_update(pc, actual)
    
    def reset_state(self):
        """Reset predictor state for new program phase"""
        self.predictor.reset()
    
    def get_accuracy(self) -> float:
        """Get prediction accuracy"""
        return self.predictor.get_accuracy()
    
    def get_mpki(self, instructions: int) -> float:
        """Get MPKI metric"""
        return self.predictor.get_mpki(instructions)

# ==========================================================
# FEDERATED LEARNING WITH GSHARE
# ==========================================================
class FederatedGShare:
    """Federated learning with GShare predictors"""
    
    def __init__(self, config: GShareConfig):
        self.config = config
        self.global_model = GShareModel(config)
        
    def train_client(self, model: GShareModel, data_loader: DataLoader):
        """
        Train client on their data stream
        GShare training is just updating state with actual outcomes
        """
        model.reset_state()  # Reset for new program phase
        
        for Xb, yb in data_loader:
            for i in range(Xb.size(0)):
                pc = Xb[i, 0].item()
                actual = yb[i].item()
                
                # Predict and update with actual outcome
                model.predict_and_update(pc, actual)
        
        return model
    
    def aggregate_models(self, client_models: List[GShareModel]):
        """
        Aggregate client models by averaging prediction table counters
        This is a simple federated averaging approach for GShare
        """
        # Create new aggregated model
        agg_model = GShareModel(self.config)
        num_clients = len(client_models)
        
        # Average prediction table entries
        for i in range(self.config.table_size):
            sum_counter = 0
            for client_model in client_models:
                sum_counter += client_model.predictor.prediction_table[i]
            
            # Average and round to nearest integer (counters must be integers)
            avg_counter = int(round(sum_counter / num_clients))
            # Clamp to valid range [0, 3]
            avg_counter = max(0, min(3, avg_counter))
            agg_model.predictor.prediction_table[i] = avg_counter
        
        # Average global history (though this might not be ideal, we'll do it anyway)
        sum_history = 0
        for client_model in client_models:
            sum_history += client_model.predictor.global_history
        agg_model.predictor.global_history = int(round(sum_history / num_clients))
        agg_model.predictor.global_history &= agg_model.predictor.history_mask
        
        return agg_model

# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================
def evaluate_predictor(predictor: GSharePredictor, data_loader: DataLoader) -> Tuple[float, float]:
    """
    Evaluate predictor accuracy and MPKI on dataset
    Returns: (accuracy, mpki)
    """
    # Create a fresh predictor for evaluation
    eval_predictor = GSharePredictor(predictor.config)
    
    correct = 0
    total = 0
    mispredictions = 0
    
    for Xb, yb in data_loader:
        for i in range(Xb.size(0)):
            pc = Xb[i, 0].item()
            actual = yb[i].item()
            
            # Predict
            pred = eval_predictor.predict(pc)
            
            # Update with actual
            eval_predictor.update(pc, actual)
            
            # Count
            if pred == actual:
                correct += 1
            else:
                mispredictions += 1
            total += 1
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    mpki = (mispredictions / total) * 1000 if total > 0 else 0
    
    return accuracy, mpki

# ==========================================================
# LOAD CLIENT DATA
# ==========================================================
train_files = sorted([f for f in os.listdir(DATA_DIR) if "train" in f])

client_train_loaders = []
client_test_loaders = []
client_test_labels = []

for f in train_files:
    X_train, y_train = load_csv(os.path.join(DATA_DIR, f))
    X_test, y_test = load_csv(os.path.join(DATA_DIR, f.replace("train", "test")))
    
    client_train_loaders.append(make_loader(X_train, y_train, shuffle=True))
    client_test_loaders.append(make_loader(X_test, y_test, shuffle=False))
    client_test_labels.append(y_test)

# Global test
X_global, y_global = load_csv(os.path.join(DATA_DIR, "test.csv"))
global_loader = make_loader(X_global, y_global, shuffle=False)

num_clients = len(client_train_loaders)

# ==========================================================
# INITIALIZE GSHARE PREDICTOR
# ==========================================================
# Try different GShare configurations
configs_to_try = [
    ("Small (1KB)", GShareConfig(table_size=1024, history_bits=10)),
    ("Medium (4KB)", GShareConfig(table_size=4096, history_bits=12)),
    ("Large (16KB)", GShareConfig(table_size=16384, history_bits=14)),
]

# Use medium configuration by default
config = GShareConfig(table_size=4096, history_bits=12)

print(f"\n{'='*60}")
print(f"GSHARE BRANCH PREDICTOR CONFIGURATION")
print(f"{'='*60}")
print(f"Table size: {config.table_size} entries")
print(f"History bits: {config.history_bits}")
print(f"Counter bits: {config.counter_bits}")
print(f"Total storage: {config.get_total_storage_kb():.2f} KB")
print(f"Total storage: {config.get_total_storage_bytes()} bytes")
print(f"Table index bits: {int(np.log2(config.table_size))} bits")

federated = FederatedGShare(config)

# ==========================================================
# FEDERATED LEARNING LOOP
# ==========================================================
print(f"\n{'='*60}")
print(f"Starting Federated Learning with GShare")
print(f"{'='*60}")

global_accuracies = []
global_mpki_list = []

for rnd in range(1, FL_ROUNDS + 1):
    print(f"\n{'='*40}")
    print(f"ROUND {rnd}/{FL_ROUNDS}")
    print(f"{'='*40}")
    
    client_models = []
    client_accuracies = []
    client_mpki_list = []
    
    # -------------------
    # CLIENT TRAINING
    # -------------------
    for c in range(num_clients):
        # Create client model
        client_model = GShareModel(config)
        
        # Train on client data stream
        trained_model = federated.train_client(client_model, client_train_loaders[c])
        
        # Evaluate on client test set
        accuracy, mpki = evaluate_predictor(trained_model.predictor, client_test_loaders[c])
        client_accuracies.append(accuracy)
        client_mpki_list.append(mpki)
        
        client_models.append(trained_model)
        
        print(f"Client {c:2d} | Test Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")
    
    # -------------------
    # AVERAGE CLIENT STATS
    # -------------------
    avg_acc = np.mean(client_accuracies)
    avg_mpki = np.mean(client_mpki_list)
    print(f"\nAvg Client | Acc: {avg_acc:.2f}% | MPKI: {avg_mpki:.2f}")
    
    # -------------------
    # FEDERATED AGGREGATION
    # -------------------
    global_model = federated.aggregate_models(client_models)
    
    # -------------------
    # GLOBAL TEST EVALUATION
    # -------------------
    global_accuracy, global_mpki = evaluate_predictor(global_model.predictor, global_loader)
    global_accuracies.append(global_accuracy)
    global_mpki_list.append(global_mpki)
    
    print(f"\n>>> Global Model | Acc: {global_accuracy:.2f}% | MPKI: {global_mpki:.2f}")
    
    # Update global model for next round
    federated.global_model = global_model

# ==========================================================
# FINAL EVALUATION
# ==========================================================
print(f"\n{'='*60}")
print(f"FINAL EVALUATION")
print(f"{'='*60}")

# Test on all client test sets with final global model
print("\nPer-client evaluation with final global model:")
client_final_accs = []
client_final_mpki = []

for c in range(num_clients):
    accuracy, mpki = evaluate_predictor(global_model.predictor, client_test_loaders[c])
    client_final_accs.append(accuracy)
    client_final_mpki.append(mpki)
    print(f"Client {c:2d} | Acc: {accuracy:.2f}% | MPKI: {mpki:.2f}")

print(f"\n{'='*40}")
print(f"CLIENT AVERAGES")
print(f"{'='*40}")
print(f"Average Client Accuracy: {np.mean(client_final_accs):.2f}% ± {np.std(client_final_accs):.2f}")
print(f"Average Client MPKI: {np.mean(client_final_mpki):.2f} ± {np.std(client_final_mpki):.2f}")

# Global test set evaluation
final_accuracy, final_mpki = evaluate_predictor(global_model.predictor, global_loader)
print(f"\n{'='*40}")
print(f"GLOBAL RESULTS")
print(f"{'='*40}")
print(f"Final Global Accuracy: {final_accuracy:.2f}%")
print(f"Final Global MPKI: {final_mpki:.2f}")

# Detailed statistics
total_samples = len(y_global)
correct = int(final_accuracy * total_samples / 100)
print(f"\nDetailed Statistics:")
print(f"  Total samples: {total_samples}")
print(f"  Correct predictions: {correct}")
print(f"  Mispredictions: {total_samples - correct}")

# ==========================================================
# PERFORMANCE METRICS
# ==========================================================
print(f"\n{'='*60}")
print(f"PERFORMANCE SUMMARY")
print(f"{'='*60}")

print(f"\nGShare Predictor Metrics:")
print(f"  Storage: {config.get_total_storage_kb():.2f} KB ({config.get_total_storage_bytes()} bytes)")
print(f"  Table entries: {config.table_size}")
print(f"  Global history bits: {config.history_bits}")
print(f"  Counter bits: {config.counter_bits} (2-bit saturating)")
print(f"  Prediction latency: 1-2 cycles (table lookup + XOR)")

print(f"\nAccuracy Metrics:")
print(f"  Final Global Accuracy: {final_accuracy:.2f}%")
print(f"  Final Global MPKI: {final_mpki:.2f}")
print(f"  Average Client Accuracy: {np.mean(client_final_accs):.2f}%")
print(f"  Average Client MPKI: {np.mean(client_final_mpki):.2f}")

# Compare different configurations
print(f"\n{'='*60}")
print(f"CONFIGURATION COMPARISON")
print(f"{'='*60}")

for name, cfg in configs_to_try:
    print(f"\n{name}:")
    print(f"  Table size: {cfg.table_size} entries")
    print(f"  History bits: {cfg.history_bits}")
    print(f"  Storage: {cfg.get_total_storage_kb():.2f} KB")
    print(f"  Index bits: {int(np.log2(cfg.table_size))} bits")

# ==========================================================
# SAVE MODEL (OPTIONAL)
# ==========================================================
model_path = "gshare_final.pth"
torch.save({
    'config': {
        'table_size': config.table_size,
        'history_bits': config.history_bits,
        'counter_bits': config.counter_bits
    },
    'prediction_table': global_model.predictor.prediction_table,
    'global_history': global_model.predictor.global_history,
    'final_accuracy': final_accuracy,
    'final_mpki': final_mpki
}, model_path)
print(f"\nModel saved to: {model_path}")

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")

# ==========================================================
# ADDITIONAL ANALYSIS
# ==========================================================
def analyze_prediction_table(predictor: GSharePredictor):
    """Analyze the prediction table distribution"""
    table = predictor.prediction_table
    
    counts = {
        "Strongly Not Taken (0)": table.count(0),
        "Weakly Not Taken (1)": table.count(1),
        "Weakly Taken (2)": table.count(2),
        "Strongly Taken (3)": table.count(3)
    }
    
    print(f"\n{'='*60}")
    print(f"PREDICTION TABLE ANALYSIS")
    print(f"{'='*60}")
    for state, count in counts.items():
        percentage = (count / len(table)) * 100
        print(f"  {state}: {count:6d} entries ({percentage:5.2f}%)")

# Analyze final predictor state
analyze_prediction_table(global_model.predictor)

# Show convergence
print(f"\n{'='*60}")
print(f"CONVERGENCE ANALYSIS")
print(f"{'='*60}")
print(f"Global accuracy over rounds:")
for i, acc in enumerate(global_accuracies[::10]):  # Show every 10th round
    print(f"  Round {(i+1)*10:2d}: {acc:.2f}%")
print(f"  Final: {global_accuracies[-1]:.2f}%")


GSHARE BRANCH PREDICTOR CONFIGURATION
Table size: 4096 entries
History bits: 12
Counter bits: 2
Total storage: 1.00 KB
Total storage: 1025 bytes
Table index bits: 12 bits

Starting Federated Learning with GShare

ROUND 1/50
Client  0 | Test Acc: 74.15% | MPKI: 258.51
Client  1 | Test Acc: 79.52% | MPKI: 204.84
Client  2 | Test Acc: 67.12% | MPKI: 328.82
Client  3 | Test Acc: 50.07% | MPKI: 499.32
Client  4 | Test Acc: 59.71% | MPKI: 402.85
Client  5 | Test Acc: 54.76% | MPKI: 452.44

Avg Client | Acc: 64.22% | MPKI: 357.80

>>> Global Model | Acc: 65.14% | MPKI: 348.59

ROUND 2/50
Client  0 | Test Acc: 74.15% | MPKI: 258.51
Client  1 | Test Acc: 79.52% | MPKI: 204.84
Client  2 | Test Acc: 67.12% | MPKI: 328.82
Client  3 | Test Acc: 50.07% | MPKI: 499.32
Client  4 | Test Acc: 59.71% | MPKI: 402.85
Client  5 | Test Acc: 54.76% | MPKI: 452.44

Avg Client | Acc: 64.22% | MPKI: 357.80

>>> Global Model | Acc: 65.14% | MPKI: 348.59

ROUND 3/50
Client  0 | Test Acc: 74.15% | MPKI: 258.51
Cli

KeyboardInterrupt: 